# ☁️ Módulo 5: Despliegue de Modelos de Machine Learning
## 19. Despliegue en Plataformas Cloud (AWS, GCP, Azure)

### Curso: **Machine Learning con Python** (IFCD093PO)
**Duración estimada:** 4 horas

---

## 🎯 Objetivos del Notebook

Hemos creado una API y la hemos empaquetado en un contenedor Docker. Ahora, ¿dónde la ejecutamos para que esté disponible globalmente, sea escalable y fiable? La respuesta es: **la nube**.

En este notebook, exploraremos conceptualmente cómo desplegar nuestro contenedor Docker en las tres principales plataformas de nube pública: **Amazon Web Services (AWS)**, **Google Cloud Platform (GCP)** y **Microsoft Azure**. No realizaremos un despliegue real, ya que requiere configurar cuentas, facturación y herramientas de línea de comandos específicas, pero entenderás el proceso y los servicios clave involucrados.

**Objetivos principales:**
1.  Entender el concepto de **Registro de Contenedores** (Container Registry).
2.  Conocer los principales servicios de **cómputo para contenedores** en cada proveedor de nube.
3.  Comparar las diferentes opciones: IaaS, PaaS, FaaS (Serverless).
4.  Esbozar los pasos generales para desplegar nuestra API `rent-predictor-api` en la nube.

---

## 📚 Contenidos del Notebook

1. [El Flujo de Trabajo del Despliegue en la Nube](#1-flujo)
2. [Paso 1: El Registro de Contenedores](#2-registry)
   - [¿Qué es y por qué lo necesitamos?](#2.1-que-es)
   - [Registros en los Proveedores Cloud](#2.2-proveedores)
3. [Paso 2: Elegir un Servicio de Cómputo](#3-computo)
   - [Opción A: Máquinas Virtuales (IaaS - Infrastructure as a Service)](#3.1-iaas)
   - [Opción B: Plataformas de Contenedores Gestionadas (PaaS - Platform as a Service)](#3.2-paas)
   - [Opción C: Funciones como Servicio / Serverless (FaaS - Function as a Service)](#3.3-faas)
4. [Guía de Despliegue Conceptual por Proveedor](#4-guias)
   - [Despliegue en Google Cloud (GCP)](#4.1-gcp)
   - [Despliegue en AWS](#4.2-aws)
   - [Despliegue en Microsoft Azure](#4.3-azure)
5. [Monitorización y Gestión](#5-monitorizacion)
6. [Ejemplo Práctico: Despliegue con Streamlit](#6-streamlit)
   - [¿Por qué Streamlit para ML?](#6.1-porque-streamlit)
   - [Estructura Básica de una App](#6.2-estructura)
   - [Ejemplo Práctico: App de Predicción](#6.3-ejemplo)
   - [Despliegue en Streamlit Cloud](#6.4-deploy)
   - [Características Avanzadas](#6.5-avanzado)
   - [Comparación de Opciones](#6.6-comparacion)
   - [Mejores Prácticas](#6.7-mejores)
   - [Dashboard de Análisis](#6.8-dashboard)
   - [Recursos Adicionales](#6.9-recursos)
   - [Conclusión](#6.10-conclusion)
7. [Resumen y Próximos Pasos](#7-resumen)

---

## 1. El Flujo de Trabajo del Despliegue en la Nube <a id='1-flujo'></a>

Independientemente del proveedor de nube que elijas, el proceso general para desplegar una aplicación contenerizada es casi siempre el mismo:

1.  **Construir la Imagen Docker**: Ya lo hicimos en el notebook anterior con `docker build`. Tenemos nuestra imagen `rent-predictor-api` localmente.

2.  **Subir la Imagen a un Registro de Contenedores**: No podemos ejecutar una imagen que solo existe en nuestra máquina. Necesitamos subirla a un repositorio centralizado en la nube.

3.  **Desplegar la Imagen en un Servicio de Cómputo**: Le decimos a un servicio de la nube (como un orquestador de contenedores o una máquina virtual) que descargue la imagen desde el registro y la ejecute.

4.  **Exponer y Configurar el Acceso**: Configuramos las redes y DNS para que nuestra API sea accesible desde internet a través de una URL pública (ej. `https://api.miempresa.com/predict`).

![Cloud Deployment Flow](imagenes/cloud.png)

---

## 2. Paso 1: El Registro de Contenedores <a id='2-registry'></a>

### 2.1. ¿Qué es y por qué lo necesitamos? <a id='2.1-que-es'></a>

Un **Registro de Contenedores** es un sistema de almacenamiento y distribución para imágenes Docker. Es como un "GitHub para imágenes Docker".

Lo necesitamos porque los servicios de cómputo en la nube necesitan un lugar desde donde descargar la imagen que deben ejecutar. No pueden acceder a tu ordenador local.

### 2.2. Registros en los Proveedores Cloud <a id='2.2-proveedores'></a>

Cada proveedor de nube ofrece su propio registro de contenedores, totalmente integrado con el resto de sus servicios:

- **AWS**: **Amazon Elastic Container Registry (ECR)**
- **GCP**: **Google Artifact Registry** (anteriormente Google Container Registry - GCR)
- **Azure**: **Azure Container Registry (ACR)**

También existe **Docker Hub**, que es el registro público más conocido, pero para aplicaciones privadas y profesionales, se suele usar el registro del propio proveedor de nube.

---

## 3. Paso 2: Elegir un Servicio de Cómputo <a id='3-computo'></a>

Una vez que nuestra imagen está en un registro, necesitamos "un ordenador" que la ejecute. Aquí es donde las cosas se ponen interesantes, ya que hay diferentes niveles de abstracción.

### 3.1. Opción A: Máquinas Virtuales (IaaS) <a id='3.1-iaas'></a>

- **Concepto**: Alquilas una máquina virtual (un servidor) en la nube. Tienes control total sobre ella: instalas Docker, descargas la imagen y la ejecutas.
- **Servicios**: 
    - **AWS**: EC2 (Elastic Compute Cloud)
    - **GCP**: Compute Engine
    - **Azure**: Virtual Machines
- **Ventajas**: Máximo control y flexibilidad.
- **Desventajas**: **Mucha gestión manual**. Eres responsable de la seguridad, las actualizaciones del SO, la configuración de red, etc. No es la opción recomendada para empezar.

### 3.2. Opción B: Plataformas de Contenedores Gestionadas (PaaS) <a id='3.2-paas'></a>

- **Concepto**: Le dices a la plataforma: "ejecuta esta imagen de contenedor, asegúrate de que siempre haya X copias corriendo y escala automáticamente si hay mucho tráfico". La plataforma se encarga de la infraestructura subyacente.
- **Servicios (Opción Serverless/Simplificada - ¡Recomendada para empezar!)**:
    - **AWS**: **AWS App Runner** o **AWS Fargate** con ECS.
    - **GCP**: **Google Cloud Run**.
    - **Azure**: **Azure Container Apps**.
- **Servicios (Opción con Orquestación Completa - para sistemas complejos)**:
    - **AWS**: EKS (Elastic Kubernetes Service)
    - **GCP**: GKE (Google Kubernetes Engine)
    - **Azure**: AKS (Azure Kubernetes Service)
- **Ventajas**: **El punto ideal para la mayoría de las APIs de ML**. Abstrae la complejidad de los servidores, es escalable, gestionado y rentable (pagas por uso).
- **Desventajas**: Menos control que una VM, pero más que suficiente para la mayoría de los casos.

### 3.3. Opción C: Funciones como Servicio / Serverless (FaaS) <a id='3.3-faas'></a>

- **Concepto**: Despliegas solo tu código (una función) sin preocuparte por contenedores o servidores. La plataforma lo gestiona todo. Recientemente, estos servicios también han añadido soporte para desplegar contenedores directamente.
- **Servicios**:
    - **AWS**: **AWS Lambda** (ahora soporta imágenes de contenedor).
    - **GCP**: **Google Cloud Functions** (también soporta contenedores).
    - **Azure**: **Azure Functions**.
- **Ventajas**: Máxima abstracción, paga solo por milisegundo de ejecución, escala a cero (no pagas si no se usa).
- **Desventajas**: Limitaciones en el tiempo de ejecución y tamaño del paquete. Puede ser más complejo para dependencias pesadas como las de ML, aunque ha mejorado mucho.

---

## 4. Guía de Despliegue Conceptual por Proveedor <a id='4-guias'></a>

A continuación, se esbozan los pasos para desplegar nuestra API `rent-predictor-api` usando la **opción recomendada (PaaS simplificado)** en cada nube.

### 4.1. Despliegue en Google Cloud (GCP) con Cloud Run <a id='4.2-gcp'></a>

**Google Cloud Run** es a menudo considerado el servicio más sencillo para empezar.

1.  **Autenticación**: Instala la CLI de gcloud (`gcloud`) y configúrala (`gcloud auth login`, `gcloud config set project [PROJECT_ID]`).
2.  **Subir la Imagen**: 
    - Habilita la API de Artifact Registry.
    - Configura Docker para autenticarse con gcloud: `gcloud auth configure-docker`.
    - Etiqueta tu imagen: `docker tag rent-predictor-api gcr.io/[PROJECT_ID]/rent-predictor-api:v1`.
    - Súbela: `docker push gcr.io/[PROJECT_ID]/rent-predictor-api:v1`.
3.  **Desplegar en Cloud Run**:
    - Ve a la consola de Cloud Run o usa la CLI:
      ```bash
      gcloud run deploy rent-predictor-service \
        --image gcr.io/[PROJECT_ID]/rent-predictor-api:v1 \
        --platform managed \
        --region europe-west1 \
        --allow-unauthenticated
      ```
4.  **Resultado**: GCP te proporcionará una URL HTTPS pública donde tu API estará disponible.

### 4.2. Despliegue en AWS con App Runner <a id='4.1-aws'></a>

**AWS App Runner** es la respuesta de Amazon a la simplicidad de Cloud Run.

1.  **Autenticación**: Instala la CLI de AWS (`aws-cli`) y configúrala (`aws configure`).
2.  **Crear Repositorio en ECR**: Crea un repositorio en Amazon ECR llamado `rent-predictor-api`.
3.  **Subir la Imagen**:
    - Obtén el comando de login de ECR: `aws ecr get-login-password --region [REGION] | docker login --username AWS --password-stdin [ACCOUNT_ID].dkr.ecr.[REGION].amazonaws.com`.
    - Etiqueta tu imagen: `docker tag rent-predictor-api:latest [ACCOUNT_ID].dkr.ecr.[REGION].amazonaws.com/rent-predictor-api:latest`.
    - Súbela: `docker push [ACCOUNT_ID].dkr.ecr.[REGION].amazonaws.com/rent-predictor-api:latest`.
4.  **Desplegar en App Runner**:
    - Ve a la consola de AWS App Runner.
    - Crea un nuevo servicio, selecciona "Container registry" como fuente.
    - Elige "Amazon ECR" y busca tu imagen `rent-predictor-api`.
    - Configura el puerto (8000) y despliega.
5.  **Resultado**: App Runner te dará una URL HTTPS pública para tu API.

### 4.3. Despliegue en Microsoft Azure con Container Apps <a id='4.3-azure'></a>

**Azure Container Apps** es el servicio equivalente en Azure, construido sobre Kubernetes pero con una experiencia simplificada.

1.  **Autenticación**: Instala la CLI de Azure (`az`) y loguéate (`az login`).
2.  **Crear Registro en ACR**: Crea un Azure Container Registry (ACR).
3.  **Subir la Imagen**:
    - Loguéate en tu ACR: `az acr login --name [ACR_NAME]`.
    - Etiqueta tu imagen: `docker tag rent-predictor-api [ACR_NAME].azurecr.io/rent-predictor-api:v1`.
    - Súbela: `docker push [ACR_NAME].azurecr.io/rent-predictor-api:v1`.
4.  **Desplegar en Container Apps**:
    - Crea un "Container App Environment" (un entorno de ejecución).
    - Crea una nueva Container App, apuntando a la imagen en tu ACR.
    - Configura el "Ingress" para permitir tráfico externo en el puerto 8000.
5.  **Resultado**: Azure te proporcionará la URL pública de la aplicación.

---

## 5. Monitorización y Gestión <a id='5-monitorizacion'></a>

Una vez desplegado, el trabajo no ha terminado. Es crucial monitorizar la aplicación:

- **Logs de la Aplicación**: Todos estos servicios te permiten ver los logs de tu API (los `print` o logs de `uvicorn`) para depurar errores. 
  - **Servicios**: AWS CloudWatch, Google Cloud Logging, Azure Monitor.

- **Métricas del Modelo (MLOps)**: ¿Sigue siendo preciso el modelo? ¿Ha cambiado la distribución de los datos de entrada (model drift)? Esto requiere herramientas de MLOps más avanzadas que se integrarían con nuestra API para registrar las predicciones y analizarlas.

- **Métricas de Rendimiento de la API**: ¿Responde rápido? ¿Da errores 500? Las plataformas cloud ofrecen dashboards para monitorizar la latencia, el número de peticiones, el uso de CPU, etc.

---

## 6. Ejemplo Práctico: Despliegue con Streamlit <a id='6-streamlit'></a>

Hasta ahora hemos visto cómo desplegar APIs en infraestructuras cloud profesionales. Sin embargo, existe una alternativa **extremadamente sencilla y visual** para poner tus modelos de ML en producción: **Streamlit**.

**Streamlit** es un framework de Python que permite crear aplicaciones web interactivas con muy pocas líneas de código. Es perfecto para:
- 📊 **Prototipos rápidos** de aplicaciones de ML
- 🎨 **Demos interactivas** de modelos para stakeholders
- 🚀 **Despliegue rápido** sin necesidad de conocimientos de frontend
- 👥 **Aplicaciones internas** para equipos de Data Science

### 6.1. ¿Por qué Streamlit para ML? <a id='6.1-porque-streamlit'></a>

**Ventajas principales:**
- ✅ **Código 100% Python**: No necesitas HTML, CSS o JavaScript
- ✅ **Componentes interactivos** integrados: sliders, botones, selectores, gráficos
- ✅ **Hot-reload**: Los cambios se reflejan automáticamente en el navegador
- ✅ **Despliegue gratuito**: Streamlit Community Cloud permite despliegues públicos gratis
- ✅ **Integración con ML**: Funciona perfectamente con scikit-learn, TensorFlow, PyTorch, etc.

**Diferencias con FastAPI:**
- **FastAPI**: Crea APIs REST (endpoints JSON) → Ideal para integrar con otras aplicaciones
- **Streamlit**: Crea interfaces web interactivas → Ideal para demos y aplicaciones de usuario final

### 6.2. Estructura Básica de una App Streamlit <a id='6.2-estructura'></a>

Una aplicación Streamlit típica tiene esta estructura:

```
my_ml_app/
├── app.py                    # Código principal de Streamlit
├── model/
│   └── trained_model.pkl     # Modelo entrenado
├── requirements.txt          # Dependencias
└── README.md                 # Documentación
```

El archivo `app.py` contiene toda la lógica de la aplicación y la interfaz de usuario.

### 6.3. Ejemplo Práctico: App de Predicción de Precios de Alquiler <a id='6.3-ejemplo'></a>

Vamos a crear una aplicación Streamlit completa para nuestro modelo de predicción de alquileres.

#### 📝 Paso 1: Instalar Streamlit

```bash
pip install streamlit
```

#### 📝 Paso 2: Crear el archivo `app.py`

Aquí está el código completo de nuestra aplicación:

In [ ]:
# Archivo: app.py
# Explicación del código:
# Esta aplicación web utiliza Streamlit para predecir el precio de alquiler
# mensual de una vivienda basándose en sus características. El usuario puede
# ajustar los parámetros de la vivienda a través de una barra lateral y obtener
# una predicción del precio junto con visualizaciones interactivas.

import streamlit as st
import pandas as pd
import numpy as np
import joblib
import plotly.express as px
from datetime import datetime

# Configuración de la página
# Explicación del código:
# Aquí se configura la apariencia y el diseño inicial de la aplicación Streamlit,
st.set_page_config(
    page_title="Predictor de Alquileres 🏠",
    page_icon="🏠",
    layout="wide",
    initial_sidebar_state="expanded"
)

# Título y descripción
st.title("🏠 Predictor de Precios de Alquiler")
st.markdown("""
Esta aplicación utiliza **Machine Learning** para predecir el precio de alquiler mensual 
basándose en las características de la vivienda.
""")

# Cargar el modelo (con caché para no recargarlo en cada interacción)
# Explicación del código:
# Esta función carga el modelo entrenado desde un archivo en disco y utiliza
# la caché de Streamlit para evitar recargas innecesarias en cada interacción del usuario
@st.cache_resource
def load_model():
    """Carga el modelo entrenado desde disco"""
    try:
        model = joblib.load('model/rent_predictor_model.pkl')
        return model
    except FileNotFoundError:
        st.error("❌ No se encontró el modelo. Asegúrate de que 'model/rent_predictor_model.pkl' existe.")
        return None

model = load_model()

# Sidebar para inputs del usuario
# Explicación del código:
# Aquí se crea una barra lateral donde el usuario puede ajustar las características
st.sidebar.header("📊 Características de la Vivienda")
st.sidebar.markdown("Ajusta los valores para hacer una predicción:")

# Inputs del usuario
# Explicación del código:
# Estos widgets permiten al usuario introducir las características de la vivienda
superficie = st.sidebar.slider(
    "Superficie (m²)", 
    min_value=20, 
    max_value=300, 
    value=75, 
    step=5,
    help="Tamaño total de la vivienda en metros cuadrados"
)

habitaciones = st.sidebar.number_input(
    "Número de habitaciones", 
    min_value=1, 
    max_value=10, 
    value=2,
    help="Cantidad de dormitorios"
)

banos = st.sidebar.number_input(
    "Número de baños", 
    min_value=1, 
    max_value=5, 
    value=1,
    help="Cantidad de cuartos de baño completos"
)

planta = st.sidebar.selectbox(
    "Planta del edificio",
    options=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    index=2,
    help="Piso en el que se encuentra la vivienda (0 = planta baja)"
)

tiene_ascensor = st.sidebar.checkbox(
    "¿Tiene ascensor?", 
    value=True,
    help="Indica si el edificio cuenta con ascensor"
)

parking = st.sidebar.checkbox(
    "¿Incluye parking?", 
    value=False,
    help="Plaza de garaje incluida"
)

zona = st.sidebar.selectbox(
    "Zona de la ciudad",
    options=["Centro", "Norte", "Sur", "Este", "Oeste"],
    help="Ubicación general de la vivienda"
)

# Botón de predicción para ejecutar el modelo
predict_button = st.sidebar.button("🔮 Predecir Precio", type="primary", use_container_width=True)

# Layout de dos columnas para resultados
col1, col2 = st.columns([1, 1])

# Mostrar resumen de características y predicción
with col1:
    st.subheader("📋 Resumen de Características")
    
    # Crear DataFrame para mostrar
    features_df = pd.DataFrame({
        "Característica": [
            "Superficie", "Habitaciones", "Baños", "Planta",
            "Ascensor", "Parking", "Zona"
        ],
        "Valor": [
            f"{superficie} m²",
            habitaciones,
            banos,
            planta,
            "Sí" if tiene_ascensor else "No",
            "Sí" if parking else "No",
            zona
        ]
    })
    
    st.dataframe(features_df, use_container_width=True, hide_index=True)
# Explicación del código:
# En esta sección se muestra un resumen de las características introducidas por el usuario.
with col2:
    st.subheader("💰 Predicción del Precio")
    
    if predict_button and model is not None:
        # Preparar datos para predicción
        # Nota: Ajusta esto según las features reales de tu modelo
        zona_mapping = {"Centro": 0, "Norte": 1, "Sur": 2, "Este": 3, "Oeste": 4}
        
        input_data = pd.DataFrame({
            'superficie': [superficie],
            'habitaciones': [habitaciones],
            'banos': [banos],
            'planta': [planta],
            'ascensor': [1 if tiene_ascensor else 0],
            'parking': [1 if parking else 0],
            'zona': [zona_mapping[zona]]
        })
        
        # Hacer predicción
        try:
            prediction = model.predict(input_data)[0]
            
            # Mostrar resultado con estilo
            st.metric(
                label="Precio de Alquiler Mensual Estimado",
                value=f"{prediction:,.0f} €",
                delta=f"±{prediction * 0.1:.0f} € (margen de error estimado)"
            )
            
            # Información adicional
            st.success("✅ Predicción realizada con éxito")
            
            # Comparación con rangos
            st.markdown("---")
            st.markdown("**📊 Análisis comparativo:**")
            
            precio_por_m2 = prediction / superficie
            col_a, col_b, col_c = st.columns(3)
            
            with col_a:
                st.metric("Precio por m²", f"{precio_por_m2:.2f} €/m²")
            with col_b:
                st.metric("Precio anual", f"{prediction * 12:,.0f} €")
            with col_c:
                coste_por_habitacion = prediction / habitaciones
                st.metric("Coste por habitación", f"{coste_por_habitacion:.0f} €")
                
        except Exception as e:
            st.error(f"❌ Error al hacer la predicción: {str(e)}")
    else:
        st.info("👈 Ajusta los parámetros en el panel lateral y haz clic en 'Predecir Precio'")

# Sección de visualización para análisis de factores
st.markdown("---")
st.subheader("📈 Visualización de Factores")

# Crear datos de ejemplo para visualización
if predict_button and model is not None:
    # Análisis de sensibilidad: ¿Cómo cambia el precio con la superficie?
    superficies_test = np.arange(30, 200, 10)
    precios_test = []
    
    for sup in superficies_test:
        test_data = pd.DataFrame({
            'superficie': [sup],
            'habitaciones': [habitaciones],
            'banos': [banos],
            'planta': [planta],
            'ascensor': [1 if tiene_ascensor else 0],
            'parking': [1 if parking else 0],
            'zona': [zona_mapping[zona]]
        })
        precios_test.append(model.predict(test_data)[0])
    
    # Gráfico interactivo
    fig = px.line(
        x=superficies_test, 
        y=precios_test,
        labels={'x': 'Superficie (m²)', 'y': 'Precio Estimado (€)'},
        title='Relación entre Superficie y Precio (manteniendo otras variables constantes)'
    )
    
    # Marcar el punto actual
    fig.add_scatter(
        x=[superficie], 
        y=[prediction],
        mode='markers',
        marker=dict(size=15, color='red'),
        name='Tu vivienda'
    )
    
    st.plotly_chart(fig, use_container_width=True)

# Footer
st.markdown("---")
st.markdown("""
<div style='text-align: center; color: gray;'>
    <p>🤖 Modelo de Machine Learning | 📊 Datos sintéticos para demostración</p>
    <p>Creado con Streamlit 🎈 | IFCD093PO - Machine Learning con Python</p>
</div>
""", unsafe_allow_html=True)

#### 📝 Paso 3: Crear `requirements.txt`

```txt
streamlit==1.28.0
pandas==2.0.3
numpy==1.24.3
scikit-learn==1.3.0
joblib==1.3.2
plotly==5.17.0
```

#### 🚀 Paso 4: Ejecutar la Aplicación Localmente

Para ejecutar la aplicación en tu máquina local:

```bash
streamlit run app.py
```

Esto abrirá automáticamente tu navegador en `http://localhost:8501` donde verás la aplicación corriendo.

**Características de la app:**
- 🎛️ **Controles interactivos** en el sidebar
- 📊 **Visualización** de las características seleccionadas
- 💰 **Predicción en tiempo real** del precio
- 📈 **Gráfico interactivo** de sensibilidad (cómo varía el precio con la superficie)
- 🎨 **Diseño profesional** con métricas y estilos personalizados

### 6.4. Despliegue en Streamlit Community Cloud <a id='6.4-deploy'></a>

**Streamlit Community Cloud** es una plataforma gratuita para desplegar aplicaciones Streamlit públicas.

#### 📝 Pasos para el Despliegue:

1. **Sube tu código a GitHub**:
   ```bash
   git init
   git add .
   git commit -m "Initial commit: Rent predictor app"
   git remote add origin https://github.com/tu-usuario/rent-predictor-app.git
   git push -u origin main
   ```

2. **Ve a Streamlit Community Cloud**:
   - Accede a https://streamlit.io/cloud
   - Inicia sesión con tu cuenta de GitHub

3. **Crear Nueva App**:
   - Haz clic en "New app"
   - Selecciona tu repositorio de GitHub
   - Especifica la rama (main) y el archivo principal (app.py)
   - Haz clic en "Deploy"

4. **Configuración Adicional** (opcional):
   - Puedes añadir secretos (API keys, credenciales) en la sección "Advanced settings"
   - Configurar variables de entorno

5. **Resultado**:
   - Tu app estará disponible en: `https://tu-usuario-rent-predictor-app.streamlit.app`
   - Se actualiza automáticamente cuando haces push a GitHub
   - ¡Completamente gratis para proyectos públicos!

#### 🔒 Consideraciones Importantes:

- **Modelo en el Repositorio**: El archivo `.pkl` del modelo debe estar en el repositorio
  - ⚠️ Si es muy grande (>100MB), considera usar Git LFS o subirlo a un servicio de storage
  
- **Datos Sensibles**: No incluyas datos privados o credenciales en el código
  - Usa los "Secrets" de Streamlit Cloud para información sensible
  
- **Límites Gratuitos**:
  - 1 GB de RAM por app
  - Tiempo de ejecución ilimitado
  - Apps públicas ilimitadas
  - Apps privadas limitadas (necesitas plan de pago)

### 6.5. Características Avanzadas de Streamlit <a id='6.5-avanzado'></a>

Streamlit ofrece muchas más funcionalidades que puedes incorporar:

#### 📊 **Componentes de Visualización:**
```python
import streamlit as st
import plotly.express as px

# Gráficos interactivos
fig = px.scatter(df, x='feature1', y='feature2', color='target')
st.plotly_chart(fig)

# Mapas
st.map(locations_df)

# Tablas editables
edited_df = st.data_editor(df)
```

#### 🎨 **Elementos de UI:**
```python
# Tabs 
tab1, tab2 = st.tabs(["Predicción", "Análisis"])

# Expanders
with st.expander("Ver detalles técnicos"):
    st.write("Información adicional...")

# Progress bar
progress_bar = st.progress(0)
for i in range(100):
    progress_bar.progress(i + 1)
```

#### 📁 **Upload de Archivos:**
```python
uploaded_file = st.file_uploader("Sube un CSV", type=['csv'])
if uploaded_file is not None:
    df = pd.read_csv(uploaded_file)
    st.write(df)
```

#### 🔐 **Autenticación Básica:**
```python
# En Streamlit Cloud Secrets (settings)
import streamlit as st

password = st.text_input("Contraseña", type="password")
if password == st.secrets["app_password"]:
    st.success("Acceso concedido")
    # Tu código aquí
else:
    st.error("Contraseña incorrecta")
```

#### ⚡ **Cache para Optimización:**
```python
@st.cache_data  # Cache para datos
def load_data():
    return pd.read_csv("large_dataset.csv")

@st.cache_resource  # Cache para modelos/objetos
def load_model():
    return joblib.load("model.pkl")
```

### 6.6. Comparación: Streamlit vs FastAPI vs Cloud Platforms <a id='6.6-comparacion'></a>

| Aspecto | Streamlit | FastAPI + Docker | Cloud Platforms (AWS/GCP/Azure) |
|---------|-----------|------------------|----------------------------------|
| **Complejidad** | ⭐ Muy fácil | ⭐⭐⭐ Moderada | ⭐⭐⭐⭐⭐ Alta |
| **Código necesario** | Mínimo (~50 líneas) | Medio (~200 líneas) | Alto (infraestructura + código) |
| **Interfaz de usuario** | ✅ Incluida y bonita | ❌ Solo API (necesitas frontend) | ❌ Solo API |
| **Ideal para** | Demos, prototipos, apps internas | APIs REST, microservicios | Producción escalable empresarial |
| **Coste despliegue** | 💰 Gratis (Community Cloud) | 💰💰 Variable (hosting) | 💰💰💰 Pay-per-use |
| **Escalabilidad** | ⚠️ Limitada | ✅ Buena | ✅✅ Excelente |
| **Tiempo de deploy** | ⏱️ 5 minutos | ⏱️ 30-60 minutos | ⏱️ 1-3 horas (primera vez) |
| **Curva de aprendizaje** | 📚 1 día | 📚📚 1 semana | 📚📚📚 Varias semanas |
| **Mantenimiento** | ✅ Mínimo | ✅ Medio | ⚠️ Requiere DevOps |

#### 🎯 Cuándo usar cada opción:

**Usa Streamlit cuando:**
- 🎨 Necesitas una demo rápida para stakeholders
- 👥 Desarrollas herramientas internas para tu equipo
- 🚀 Quieres un MVP en horas, no días
- 📊 Priorizas la visualización e interactividad
- 💻 Tu audiencia son usuarios finales (no desarrolladores)

**Usa FastAPI + Docker cuando:**
- 🔌 Necesitas una API REST para integrar con otras apps
- 📱 Vas a construir un frontend separado (React, Angular, etc.)
- 🏢 Requieres control total sobre la arquitectura
- 🔄 Necesitas alta escalabilidad horizontal
- 🤖 Tus clientes son otras aplicaciones, no humanos

**Usa Cloud Platforms cuando:**
- 🌍 Necesitas alcance global y alta disponibilidad
- 📈 Esperas tráfico masivo y variable
- 🔒 Requieres cumplimiento de normativas estrictas
- 💼 Es un servicio crítico de producción empresarial
- 🛡️ Necesitas SLAs y soporte profesional

### 6.7. Mejores Prácticas para Streamlit <a id='6.7-mejores'></a>

#### ✅ DO (Haz esto):

1. **Usa caché agresivamente**:
   ```python
   @st.cache_resource
   def load_model():
       return joblib.load('model.pkl')
   ```

2. **Organiza el código con funciones**:
   ```python
   def render_sidebar():
       st.sidebar.header("Inputs")
       # ...
   
   def render_results(prediction):
       st.metric("Precio", f"{prediction}€")
       # ...
   ```

3. **Añade información contextual**:
   ```python
   st.info("ℹ️ Esta predicción tiene un margen de error del ±10%")
   ```

4. **Valida inputs del usuario**:
   ```python
   if superficie < 20:
       st.error("⚠️ La superficie debe ser mayor a 20m²")
       st.stop()
   ```

5. **Proporciona feedback visual**:
   ```python
   with st.spinner("Calculando predicción..."):
       prediction = model.predict(data)
   st.success("✅ Predicción completada")
   ```

#### ❌ DON'T (Evita esto):

1. **No cargues el modelo en cada interacción** (usa `@st.cache_resource`)
2. **No hagas la app muy lenta** (procesa solo cuando el usuario lo pida)
3. **No sobrecargues la UI** (usa tabs, expanders y columnas)
4. **No expongas información sensible** (usa Secrets)
5. **No olvides manejar errores** (try-except con mensajes claros)

### 6.8. Ejemplo Adicional: Dashboard de Análisis del Modelo <a id='6.8-dashboard'></a>

Aquí un ejemplo de cómo crear un dashboard más completo con análisis del modelo:

```python
import streamlit as st
import pandas as pd
import plotly.express as px
from sklearn.metrics import mean_absolute_error, r2_score

st.title("📊 Dashboard de Análisis del Modelo")

# Tabs para organizar
tab1, tab2, tab3 = st.tabs(["📈 Predicciones", "🎯 Métricas", "🔍 Feature Importance"])

with tab1:
    st.header("Hacer Predicción")
    # Tu código de predicción aquí
    
with tab2:
    st.header("Métricas del Modelo")
    
    # Cargar datos de test (asume que tienes X_test, y_test guardados)
    if st.button("Calcular Métricas"):
        predictions = model.predict(X_test)
        mae = mean_absolute_error(y_test, predictions)
        r2 = r2_score(y_test, predictions)
        
        col1, col2 = st.columns(2)
        col1.metric("MAE", f"{mae:.2f}€")
        col2.metric("R² Score", f"{r2:.3f}")
        
        # Gráfico de predicciones vs reales
        fig = px.scatter(
            x=y_test, 
            y=predictions,
            labels={'x': 'Precio Real', 'y': 'Precio Predicho'},
            title='Predicciones vs Valores Reales'
        )
        fig.add_shape(type="line", x0=y_test.min(), y0=y_test.min(),
                     x1=y_test.max(), y1=y_test.max())
        st.plotly_chart(fig)

with tab3:
    st.header("Importancia de Features")
    
    # Si usas un modelo tree-based
    if hasattr(model, 'feature_importances_'):
        importances = pd.DataFrame({
            'Feature': X_train.columns,
            'Importance': model.feature_importances_
        }).sort_values('Importance', ascending=False)
        
        fig = px.bar(importances, x='Importance', y='Feature', orientation='h')
        st.plotly_chart(fig)
```

Este código crea un dashboard profesional con múltiples vistas del modelo.

### 6.9. Recursos Adicionales para Streamlit <a id='6.9-recursos'></a>

#### 📚 Documentación y Tutoriales:
- **Documentación oficial**: https://docs.streamlit.io
- **Galería de apps**: https://streamlit.io/gallery
- **Cheat sheet**: https://docs.streamlit.io/library/cheatsheet
- **30 Days of Streamlit**: https://30days.streamlit.app

#### 🎨 Componentes Adicionales:
- **Streamlit-Authenticator**: Autenticación de usuarios
- **Streamlit-Aggrid**: Tablas interactivas avanzadas
- **Streamlit-Option-Menu**: Menús de navegación personalizados
- **Plotly**: Gráficos interactivos (ya lo usamos)
- **Altair**: Otra librería de visualización

#### 💡 Proyectos de Ejemplo:
- Clasificadores de imágenes con CNN
- Análisis de sentimientos de texto
- Dashboards de datos financieros
- Aplicaciones de Computer Vision en tiempo real
- Chatbots con LLMs

#### 🚀 Streamlit para MLOps:
Streamlit puede integrarse con herramientas de MLOps:
- **MLflow**: Para tracking de experimentos
- **Weights & Biases**: Para visualización de métricas
- **DVC**: Para versionado de datos y modelos

### 6.10. Conclusión: Streamlit en tu Stack de ML <a id='6.10-conclusion'></a>

**Streamlit es una herramienta poderosa que todo científico de datos debería conocer**. No reemplaza a las soluciones enterprise como AWS o Azure, pero complementa tu stack perfectamente:

#### 🎯 Flujo de Trabajo Recomendado:

1. **Prototipo con Streamlit** (días 1-3):
   - Desarrolla rápidamente una app interactiva
   - Valida la idea con stakeholders
   - Itera rápidamente basándote en feedback

2. **Si necesitas API, crea con FastAPI** (semana 1):
   - Desarrolla endpoints REST robustos
   - Documenta con OpenAPI/Swagger
   - Empaqueta en Docker

3. **Para producción enterprise, despliega en Cloud** (semanas 2-4):
   - Sube a AWS/GCP/Azure
   - Configura CI/CD
   - Implementa monitorización

**Pero muchas veces, ¡Streamlit es suficiente!** Especialmente para:
- 🏢 Herramientas internas de la empresa
- 📊 Dashboards de análisis
- 🎓 Proyectos académicos y portfolios
- 🧪 Experimentación y A/B testing
- 👥 Apps con usuarios limitados (<1000 usuarios simultáneos)

---

**💡 Recomendación final**: Empieza siempre con Streamlit. Si tu proyecto crece y necesitas más escalabilidad, entonces migra a soluciones más complejas. Muchos proyectos nunca necesitarán más que Streamlit.

---

## 7. Resumen y Próximos Pasos <a id='7-resumen'></a>

En este notebook, hemos explorado el espectro completo de opciones para desplegar modelos de Machine Learning en producción.

### ✅ Lo que hemos aprendido:

1. **Flujo de trabajo en la nube**: **Build → Push to Registry → Deploy to Compute**
2. **Servicios clave** para cada paso en AWS, GCP y Azure
3. **Plataformas de contenedores gestionadas** (Cloud Run, App Runner, Container Apps) como opción ideal para APIs profesionales
4. **Streamlit** como alternativa rápida y visual para prototipos y aplicaciones interactivas

### 🎯 Opciones de Despliegue según tu Caso de Uso:

| Necesidad | Solución Recomendada | Tiempo de Implementación |
|-----------|---------------------|--------------------------|
| **Demo rápida para stakeholders** | Streamlit Community Cloud | ⏱️ Horas |
| **Herramienta interna de equipo** | Streamlit (local o cloud) | ⏱️ 1 día |
| **Prototipo MVP** | Streamlit + validación | ⏱️ 2-3 días |
| **API REST para integración** | FastAPI + Docker | ⏱️ 1 semana |
| **Producción con tráfico moderado** | FastAPI + Cloud Run/App Runner | ⏱️ 1-2 semanas |
| **Producción empresarial escalable** | Kubernetes en AWS/GCP/Azure | ⏱️ 3-4 semanas |

### 🚀 Recomendación de Ruta de Aprendizaje:

**Fase 1 - Fundamentos (empezar aquí):**
- ✅ Crea tu primera app con Streamlit
- ✅ Despliega en Streamlit Community Cloud
- ✅ Comparte con colegas y recibe feedback

**Fase 2 - Profesionalización:**
- ✅ Aprende FastAPI y crea una API REST
- ✅ Empaqueta tu API en Docker
- ✅ Prueba localmente con Docker Compose

**Fase 3 - Producción Cloud:**
- ✅ Elige un proveedor cloud (Google Cloud Run es el más sencillo)
- ✅ Crea una cuenta gratuita
- ✅ Despliega tu primera API en la nube
- ✅ Configura dominio personalizado y SSL

**Fase 4 - MLOps Avanzado:**
- ✅ Implementa CI/CD con GitHub Actions
- ✅ Añade monitorización y alertas
- ✅ Implementa A/B testing
- ✅ Automatiza reentrenamiento

### 📚 Recursos Recomendados:

**Para Streamlit:**
- 📖 Documentación oficial: https://docs.streamlit.io
- 🎨 Galería de ejemplos: https://streamlit.io/gallery
- 🎓 30 Days of Streamlit: https://30days.streamlit.app

**Para Cloud Deployment:**
- ☁️ Google Cloud Run Quickstart: https://cloud.google.com/run/docs/quickstarts
- 🐳 Docker Official Documentation: https://docs.docker.com
- 🔧 FastAPI Deployment Guide: https://fastapi.tiangolo.com/deployment/

**Para MLOps:**
- 🤖 MLflow Documentation: https://mlflow.org
- 📊 Evidently AI (Model Monitoring): https://evidentlyai.com
- 🔄 DVC (Data Version Control): https://dvc.org

### 💡 Siguientes Pasos Prácticos:

1. **Elige un proyecto personal** de ML que quieras desplegar
2. **Crea una app Streamlit** en 1-2 horas para validar la idea
3. **Despliega en Streamlit Cloud** y comparte el enlace
4. **Si necesitas API**, evoluciona a FastAPI + Docker
5. **Cuando tengas tráfico real**, considera Cloud Platforms

### 🎓 Proyecto Final Sugerido:

Desarrolla una aplicación completa que incluya:
1. ✅ **Modelo entrenado** (usa uno de los notebooks anteriores del curso)
2. ✅ **App Streamlit** con interfaz interactiva
3. ✅ **API FastAPI** (opcional, para aprender)
4. ✅ **Desplegada en la nube** (Streamlit Cloud o Google Cloud Run)
5. ✅ **Documentación** en README.md
6. ✅ **Repositorio público** en GitHub para tu portfolio

---

En el **próximo notebook (`20_monitoreo_mantenimiento.ipynb`)**, profundizaremos en los conceptos de monitorización y mantenimiento de modelos en producción, incluyendo:
- 📊 Monitorización de métricas del modelo
- 🔍 Detección de model drift
- 🔄 Estrategias de reentrenamiento
- 📈 Logging y análisis de predicciones
- 🚨 Sistemas de alertas

**¡Has completado el módulo de despliegue!** 🎉 Ahora tienes el conocimiento para llevar tus modelos de ML desde un notebook a producción, con opciones que van desde prototipos rápidos hasta sistemas enterprise escalables.